# Make metacells for CAS

Stephen Fleming

2026.04.15

The idea here is to explore ways to "thin" the CAS vector search index. Do we need to search against all 100M cells?  If so, it takes a very long time.

If we can come up with smart ways to create a smaller index, we can save a lot of compute.

In [1]:
import anndata
import pandas as pd

In [2]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata


class ExtendedAnnData(anndata.AnnData):
    """AnnData subclass with a group-aggregation method."""

    def agg(self, func, obs_keys: list[str]) -> "ExtendedAnnData":
        """
        Aggregate the AnnData by unique combinations of categorical obs keys.

        Args:
            func: Aggregation function with signature func(array, axis=0) -> array.
                  E.g. np.mean, np.sum, np.median.
            obs_keys: Names of categorical obs columns to group by.

        Returns:
            A new ExtendedAnnData where each row represents a unique combination
            of the obs_keys values. X, layers, obsm, and numeric non-key obs
            columns are aggregated using func. var and uns are passed through
            unchanged. obsp is dropped.
        """
        if not obs_keys:
            raise ValueError("At least one obs key must be provided")

        for key in obs_keys:
            if key not in self.obs.columns:
                raise KeyError(f"obs key {key!r} not found in obs")
            if not pd.api.types.is_categorical_dtype(self.obs[key]):
                raise ValueError(
                    f"obs key {key!r} has dtype {self.obs[key].dtype!r}; "
                    "expected categorical. Convert first: adata.obs[key].astype('category')"
                )

        key_df = self.obs[obs_keys].copy()
        key_df["_row_pos"] = np.arange(self.n_obs)

        grouped = key_df.groupby(obs_keys, observed=True, sort=True)

        group_tuples: list[tuple] = []
        group_row_indices: list[np.ndarray] = []
        for name, sub_df in grouped:
            group_tuples.append(name if isinstance(name, tuple) else (name,))
            group_row_indices.append(sub_df["_row_pos"].values)

        def _to_dense(arr) -> np.ndarray:
            return arr.toarray() if sp.issparse(arr) else np.asarray(arr)

        def _agg_2d(mat: np.ndarray) -> np.ndarray:
            return np.stack([func(mat[idx], axis=0) for idx in group_row_indices])

        X_agg = _agg_2d(_to_dense(self.X))

        layers_agg = {
            name: _agg_2d(_to_dense(data))
            for name, data in self.layers.items()
        }

        obsm_agg = {
            name: _agg_2d(np.asarray(data))
            for name, data in self.obsm.items()
        }

        new_obs: dict[str, object] = {}
        for i, key in enumerate(obs_keys):
            new_obs[key] = pd.Categorical([gt[i] for gt in group_tuples])

        for col in self.obs.columns:
            if col in obs_keys:
                continue
            if pd.api.types.is_numeric_dtype(self.obs[col]):
                col_vals = self.obs[col].to_numpy()
                try:
                    new_obs[col] = [func(col_vals[idx], axis=0) for idx in group_row_indices]
                except Exception:
                    pass

        obs_df = pd.DataFrame(new_obs)

        if len(obs_keys) == 1:
            obs_df.index = pd.Index([str(gt[0]) for gt in group_tuples])
        else:
            obs_df.index = pd.Index(["_".join(str(v) for v in gt) for gt in group_tuples])

        return ExtendedAnnData(
            X=X_agg,
            obs=obs_df,
            var=self.var.copy(),
            layers=layers_agg or None,
            obsm=obsm_agg or None,
            uns=dict(self.uns),
        )

In [ ]:
column_names = [
    "dataset_id",
    "donor_id",
    "assay",
    "suspension_type",
    "cell_type",
    "cell_type_ontology_id",
    "disease",
]

In [ ]:
tiledb_experiment_uri = "gs://cellarium-dev-central/nexus_tiledb_soma/czi_census_homo_sapiens_20251108"

# get obs from tiledb and convert to pandas dataframe
import tiledbsoma

with tiledbsoma.open(tiledb_experiment_uri) as exp:
    obs_df = (
        exp.obs
        .read(coords=(soma_joinids,), column_names=["cell_type", "tissue", "donor_id"])
        .concat()
        .to_pandas()
    )

obs_df